# Load the Data

In [ ]:
#Import generic libraries
import numpy as np
import pandas as pd
import torch


In [ ]:
news = pd.read_csv('/kaggle/input/mit-ai-news-published-till-2023/articles.csv')
DOCUMENT="Article Body"

In [ ]:
#Because it is just a course we select a small portion of News.
MAX_NEWS = 3
subset_news = news.head(MAX_NEWS)

In [ ]:
subset_news.head()

,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...


In [ ]:
articles = subset_news[DOCUMENT].tolist()

# Load the Models and create the summaries

Both models are available on Hugging Face, so we will work with the Transformers library.

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name_small = "t5-base"
model_name_reference = "flax-community/t5-base-cnn-dm"
#model_name_reference = "pszemraj/long-t5-tglobal-base-16384-booksum-V11-big_patent-V2"

In [ ]:
#This function returns the tokenizer and the Model.
def get_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    return tokenizer, model


In [ ]:
tokenizer_small, model_small = get_model(model_name_small)

/opt/conda/lib/python3.10/site-packages/transformers/models/t5/tokenization_t5_fast.py:155: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxver

In [ ]:
tokenizer_reference, model_reference = get_model(model_name_reference)

With both models downloaded and ready, we create a function that will perform the summaries.

The function takes fourth parameters:

* the list of texts to summarize.
* the tokenizer.
* the model.
* the maximum length for the generated summary

In [ ]:
def create_summaries(texts_list, tokenizer, model, max_l=125):

    # We are going to add a prefix to each article to be summarized
    # so that the model knows what it should do
    prefix = "Summarize this news: "
    summaries_list = [] #Will contain all summaries

    texts_list = [prefix + text for text in texts_list]

    for text in texts_list:

        summary=""

        #calculate the encodings
        input_encodings = tokenizer(text,
                                    max_length=1024,
                                    return_tensors='pt',
                                    padding=True,
                                    truncation=True)

        # Generate summaries
        with torch.no_grad():
            output = model.generate(
                input_ids=input_encodings.input_ids,
                attention_mask=input_encodings.attention_mask,
                max_length=max_l,  # Set the maximum length of the generated summary
                num_beams=2,     # Set the number of beams for beam search
                early_stopping=True
            )

        #Decode to get the text
        summary = tokenizer.batch_decode(output, skip_special_tokens=True)

        #Add the summary to summaries list
        summaries_list += summary
    return summaries_list


To create the summaries, we call the 'create_summaries' function, passing both the news articles and the corresponding tokenizer and model.

In [ ]:
# Creating the summaries for both models.
summaries_small = create_summaries(articles,
                                  tokenizer_small,
                                  model_small)


In [ ]:
summaries_reference = create_summaries(articles,
                                      tokenizer_reference,
                                      model_reference)

In [ ]:
summaries_small

['MIT and MIT-Watson AI Lab have developed a unified framework. the system can simultaneously predict molecular properties and generate new molecules. it uses this grammar to construct viable molecules and predict their properties.',
 '\'BioAutoMATED\' is an automated machine-learning system that can select and build an appropriate model for a given dataset. it can even take care of the laborious task of data preprocessing, whittling down a months-long process to just a few hours. \'"We want to lower these barriers for a lot of folks that want to use machine learning or biology," says first co-author Jacqueline Valeri.',
 "MIT and IBM research scientists have made a computer vision model more robust by training it to work like a part of the brain that humans and other primates rely on for object recognition. 'we asked the artificial neural network to make the function of one of your inside simulated “neural” layers as similar as possible to the corresponding biological neural layer,' s

In [ ]:
summaries_reference

['Researchers created a machine-learning system that automatically learns the "language" of molecules using only a small, domain-specific dataset. The system learns to construct viable molecules and predict their properties. Computational design and Fabrication Group will be presented at the International Conference for Machine Learning.',
 "Automated machine-learning system can select and build an appropriate model for a given dataset. 'BioAutoMATED' is an automated machine-learning system. The tool includes binary classification models, multi-class classification models, and more complex neural networks.",
 "MIT and IBM researchers have found that artificial neural networks resemble the multilayered brain circuits that process visual information in humans and other primates. 'We asked it to do both of those things as well as the standard, computer vision approach,' said one expert. The network found to be more robust by training it to work like a part of the brain that humans rely on

# ROUGE
Let's install and load all the necessary libraries to conduct a ROUGE evaluation.

In [ ]:
!pip install evaluate
!pip install rouge_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
  Preparing metadata (setup.py) ... - done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24954 sha256=2500e37dbf310b269964fd5ea0887adca6837d77106e50c3fa8eca55296c6ae2
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be

In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize

#!pip install rouge

In [ ]:
#import evaluate
#from nltk.tokenize import sent_tokenize
#from rouge_score import rouge_scorer

In [ ]:
#With the function load of the library evaluate
#we create a rouge_score object
rouge_score = evaluate.load("rouge")

Calculating ROUGE is as simple as calling the *compute* function of the *rouge_score* object we created earlier. This function takes the texts to compare as arguments and a third value *use_stemmer*, which indicates whether it should use *stemmer* or full words for the comparison.

A *stemmer* is the base of the word. Transform differents forms of a word in a same base.

Some samples of steammer are:
* Jumping -> Jump.
* Running -> Run.
* Cats -> Cat.

In [ ]:
def compute_rouge_score(generated, reference):

    #We need to add '\n' to each line before send it to ROUGE
    generated_with_newlines = ["\n".join(sent_tokenize(s.strip())) for s in generated]
    reference_with_newlines = ["\n".join(sent_tokenize(s.strip())) for s in reference]

    return rouge_score.compute(
        predictions=generated_with_newlines,
        references=reference_with_newlines,
        use_stemmer=True,

    )

In [ ]:
compute_rouge_score(summaries_small, summaries_reference)

{'rouge1': 0.47018752391886715,
 'rouge2': 0.3209013209013209,
 'rougeL': 0.34330271718331423,
 'rougeLsum': 0.44692881745120555}

We can see that there is a difference between the two models when performing summarization.

For example, in ROUGE-1, the similarity is 47%, while in ROUGE-2, it's a 32%. This indicates that the results are different, with some similarities but differents enough.

However, we still don't know which model is better since we have compared them to each other and not to a reference text. But at the very least, we know that the fine-tuning process applied to the second model has significantly altered its results.

# Comparing to a Dataset with real summaries.
We are going to load the Dataset cnn_dailymail. This is a well-known dataset available in the **Datasets** library, and it suits our purpose perfectly.

Apart from the news, it also contains pre-existing summaries.

We will compare the summaries generated by the two models we are using with those from the dataset to determine which model creates summaries that are closer to the reference ones.

In [ ]:
from datasets import load_dataset

cnn_dataset = load_dataset(
    "cnn_dailymail", version="3.0.0"
)

#Get just a few news to test
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS))

sample_cnn

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset cnn_dailymail downloaded and prepared to /root/.cache/huggingface/datasets/cnn_dailymail/default/3.0.0/3cb851bf7cf5826e45d49db2863f627cba583cbc32342df7349dfe6c38060234. Subsequent calls will reuse this data.


  0%|          | 0/3 [00:00<?, ?it/s]

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 3
})

We retrieve the maximum length of the summaries to give the models the option to generate summaries of the same length, if they choose to do so.

In [ ]:
max_length = max(len(item['highlights']) for item in sample_cnn)
max_length = max_length + 10

In [ ]:
summaries_t5_base = create_summaries(sample_cnn["article"],
                                      tokenizer_small,
                                      model_small,
                                      max_l=max_length)

In [ ]:
summaries_t5_finetuned = create_summaries(sample_cnn["article"],
                                      tokenizer_reference,
                                      model_reference,
                                      max_l=max_length)

In [ ]:
#Get the real summaries from the cnn_dataset
real_summaries = sample_cnn['highlights']

Let's take a look at the generated summaries alongside the reference summaries provided by the dataset.

In [ ]:
summaries = pd.DataFrame.from_dict(
        {
            "base": summaries_t5_base,
            "finetuned": summaries_t5_finetuned,
            "reference": real_summaries,
        }
    )
summaries.head()

,base,finetuned,reference
0,"best died in hospice in Hickory, north Carolin...","Jimmie Best was ""the most constantly creative ...","James Best, who played the sheriff on ""The Duk..."
1,"""it doesn't matter what anyone says, he is pre...",Dr. Anthony Moschetto's attorney calls the all...,A lawyer for Dr. Anthony Moschetto says the ch...
2,president Barack Obama took part in a roundtab...,President Obama says climate change is a publi...,"""No challenge poses more of a public threat th..."


Now we can calculate the ROUGE scores for the two models.

In [ ]:
compute_rouge_score(summaries_t5_base, real_summaries)

{'rouge1': 0.3050834824090638,
 'rouge2': 0.07211128178870115,
 'rougeL': 0.2095520274299344,
 'rougeLsum': 0.2662418008348241}

In [ ]:
compute_rouge_score(summaries_t5_finetuned, real_summaries)

{'rouge1': 0.31659149328289443,
 'rouge2': 0.11065084340946411,
 'rougeL': 0.22002036956205442,
 'rougeLsum': 0.24877540132887144}